# 🌿 Plant Explorer — High School Edition
## Multi-metric Plant Similarity Analysis

This notebook goes beyond the middle school version by adding **colour analysis**
as a second, independent similarity metric.

You will:
- 📋 Enter plant photos in a **data table**
- 🤖 Compute **CLIP embedding similarity** (visual AI features)
- 🎨 Extract **RGB colour spectra** and **hue distributions**
- 🎯 Find **dominant colours** via k-means clustering
- 📐 Compute **colour histogram similarity** (histogram intersection)
- ⚖️ Combine both metrics into a **weighted similarity score**
- 🗺️ Map GPS locations

> **Change lines marked ✏️**


---
## 📐 Two similarity metrics

### 1 · CLIP embedding similarity
CLIP maps each image to a point in 512-dimensional space.
Cosine similarity measures the angle between two points —
1.0 means identical, ~0.6 means unrelated.

### 2 · Colour histogram similarity
We count how many pixels fall into each intensity bin (0–255)
for R, G, B channels separately — like an emission spectrum.
**Histogram intersection** measures the overlapping area:

$$\text{similarity} = \frac{1}{3}\sum_{c \in \{R,G,B\}} \sum_i \min(h_1^c(i),\; h_2^c(i))$$

Range is 0 (no overlap) to 1 (identical distributions).

### Combined score
$$\text{score} = w_{\text{CLIP}} \cdot s_{\text{CLIP}} + w_{\text{colour}} \cdot s_{\text{colour}}$$

You control the weights below.


---
## ⚙️ Setup


In [ ]:
import subprocess, sys
for pkg in ["transformers", "torch", "Pillow", "folium",
            "scikit-learn", "datascience", "scipy"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
print("✅ Ready!")


In [ ]:
%matplotlib inline
from datascience import Table
from plant_identifier import (
    check_photos, load_clip,
    show_all_plants, extract_gps, make_map,
    plot_visible_spectra, plot_color_spectra, plot_hue_spectra,
    plot_dominant_colors, compare_plants_full,
    plot_combined_results, plot_embeddings
)
load_clip()


---
## ✏️ Step 1 — Enter your plant photos

Upload photos using the file browser, then fill in the table.


In [ ]:
mystery_photo = "mystery_plant.jpg"
mystery_lat   =  39.9526
mystery_lon   = -75.1652

plants_table = Table().with_columns(
    "Name",       ["Dandelion",             "White Clover",       "Plantain"],
    "Photo",      ["plants/dandelion.jpg",  "plants/clover.jpg",  "plants/plantain.jpg"],
    "Latitude",   [39.9530,                 39.9528,              39.9522],
    "Longitude",  [-75.1648,               -75.1655,             -75.1660],
)
plants_table


In [ ]:
known_plants = [
    {"name": row.item("Name"), "path": row.item("Photo")}
    for row in plants_table.rows
]
check_photos(mystery_photo, known_plants)


### 📷 Photos


In [ ]:
show_all_plants(mystery_photo, known_plants)


---
## 🎨 Step 2 — RGB colour spectra

Each plant gets three overlaid curves — one per colour channel.
Like emission spectra in chemistry, peaks show where most pixel
energy sits in each channel.

**Prediction:** Which known plant do you think will have the most
similar spectrum to the mystery plant?


In [ ]:
plot_color_spectra(mystery_photo, known_plants)


---
## 🔬 Step 2b — Visible-light pseudo-spectrum (400–780 nm)

This plot maps each plant's R, G, B channel intensities onto the
visible-light wavelength scale using Gaussian profiles:

| Channel | Approximate peak | Colour of light |
|---------|-----------------|-----------------|
| Blue    | 460 nm          | Blue/violet     |
| Green   | 540 nm          | Green           |
| Red     | 620 nm          | Orange-red      |

The dashed lines show each channel's contribution; the solid black
curve is their sum — an approximate **reflectance spectrum**.

> **Important:** These are *educational pseudo-spectra* based on
> camera RGB values, not true spectrometer readings.
> A real plant spectrum needs a spectrometer (e.g. Ocean Optics).
> Plants also reflect strongly in near-infrared (~750–1400 nm),
> which cameras cannot see.


In [ ]:
plot_visible_spectra(mystery_photo, known_plants)


---
## 🌈 Step 3 — Hue spectra

The RGB spectrum mixes brightness and colour.
The **hue** (from HSV colour space) strips brightness out and
shows only the *colour angle* (0° = red, 120° = green, 240° = blue).

Near-grey pixels are excluded — only chromatically rich pixels count.

> **Connection to physics:** Hue is analogous to wavelength in
> visible light spectroscopy (400 nm = violet, 700 nm = red).


In [ ]:
plot_hue_spectra(mystery_photo, known_plants)


---
## 🎯 Step 4 — Dominant colour composition

K-means clustering groups all pixel colours into N clusters
and finds the centre (average) of each group.
The result is a palette of the most representative colours,
with the fraction of pixels each one represents.


In [ ]:
plot_dominant_colors(mystery_photo, known_plants, n_colors=6)


---
## ⚖️ Step 5 — Combined similarity score

### ✏️ Set your weights

`clip_weight` controls how much the AI visual embedding counts.
`1 - clip_weight` goes to colour similarity.

Try different values and see how the ranking changes!


In [ ]:
# ✏️ Adjust the balance between AI visual features and raw colour
clip_weight = 0.6   # 0.0 = colour only,  1.0 = CLIP only

result = compare_plants_full(mystery_photo, known_plants,
                             clip_weight=clip_weight)


### 📊 Multi-metric comparison chart


In [ ]:
plot_combined_results(result)


---
## 🔵 Step 6 — Embedding space (PCA)

The CLIP embeddings projected to 2D via PCA.
Closer points = more similar *visual features* (not just colour).


In [ ]:
plot_embeddings(mystery_photo, known_plants)


---
## 📍 Step 7 — GPS map


In [ ]:
all_entries = [
    {"name": "Mystery Plant", "lat_manual": mystery_lat,
     "lon_manual": mystery_lon, "path": mystery_photo}
] + [
    {"name": row.item("Name"), "lat_manual": row.item("Latitude"),
     "lon_manual": row.item("Longitude"),  "path": row.item("Photo")}
    for row in plants_table.rows
]

plant_locations = []
for entry in all_entries:
    gps = extract_gps(entry["path"]) or           {"latitude": entry["lat_manual"], "longitude": entry["lon_manual"]}
    plant_locations.append({"name": entry["name"], **gps})

make_map(plant_locations, zoom_start=15)


---
## 📝 Analysis questions

**1. Metric comparison**
Did CLIP similarity and colour similarity agree on the best match?
If they disagreed, which do you trust more — and why?

**2. Spectra interpretation**
Look at the RGB spectra. Which channel (R, G, or B) was most
different between the mystery plant and the worst match?
What biological feature might explain this?

**3. Weight sensitivity**
Change `clip_weight` to 0.2 (colour-dominant) and then 0.9 (CLIP-dominant).
Does the ranking change? Under what conditions would you trust colour more
than AI features?

**4. K-means clustering**
K-means is sensitive to `n_colors`. Set it to 3, then 10.
How does the palette change? What is lost or gained?

**5. Extension 🌟 — Histogram intersection formula**
Prove mathematically that histogram intersection gives 1.0
when both histograms are identical and 0.0 when they have no overlap.
(Hint: what does `min(h1[i], h2[i])` equal in each case?)

**6. Extension 🌟 — Design your own metric**
Could you use *texture* instead of colour? Look up
**HOG (Histogram of Oriented Gradients)** — it's in `scikit-image`.
How would you integrate it as a third metric?

---
## 🔭 Other applications of this framework

| Domain | What to photograph | Extra metric ideas |
|--------|-------------------|--------------------|
| 🪨 **Minerals** | Rock specimens | Crystallography texture (HOG) |
| 🦋 **Entomology** | Insect specimens | Wing venation patterns |
| 🌾 **Agriculture** | Crop leaves | NDVI proxy (G–R ratio) |
| 🔬 **Histology** | Stained tissue slides | Texture + spatial frequency |
| 🌊 **Water quality** | Water samples | Turbidity via blue channel |

*🌱 You've built a multi-metric image similarity pipeline from scratch.
This is the foundation of modern computer vision and remote sensing.*
